# NLP Introduction — Hands-On Demo
### From Raw Text → Cleaning → Preprocessing → Feature Extraction → (Bonus) Modeling

This notebook is a companion to the **Session 1–2: Introduction to NLP** notes.

We will:
1. Generate a small **synthetic dataset** of product-review-style sentences (with realistic noise: HTML tags, URLs, emojis, mixed case, extra punctuation).
2. **Clean** the text (remove HTML, URLs, special characters).
3. **Preprocess**: lowercase → tokenize → remove stopwords → remove punctuation.
4. Compare **Stemming vs. Lemmatization**.
5. Extract features using **Bag of Words** and **TF-IDF**.
6. (Bonus) Train a simple **Naive Bayes** sentiment classifier to see the full pipeline in action.

> Run the cells **top to bottom**. Each step builds on the previous one.


## 0. Setup

First, install/import the libraries we need. We use:
- `nltk` for tokenization, stopwords, stemming, and lemmatization
- `pandas` for tabular data handling
- `scikit-learn` for TF-IDF / Bag-of-Words vectorization and the demo classifier
- `re` and `string` (built-in) for text cleaning


In [1]:
!pip install nltk

In [2]:
# If running for the first time, uncomment the line below to install dependencies:
# !pip install nltk scikit-learn pandas --quiet

import random
import re
import string

import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Download required NLTK resources (only needs to run once per environment)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

random.seed(42)  # for reproducibility
print("Setup complete ✅")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragsb\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ragsb\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ragsb\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ragsb\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ragsb\AppData\Roaming\nltk_data...


Setup complete ✅


## 1. Text Collection — Generating Synthetic Data

In a real project, this step would involve scraping websites, calling an API, or loading a
public dataset. Here, we **synthetically generate** product review sentences using templates,
so the notebook is fully self-contained and reproducible.

We deliberately inject realistic "noise" into the text:
- HTML tags like `<br>`, `<p>`
- URLs like `http://shop.example.com/1234`
- Emojis 😍 😡
- Extra punctuation (`!!!`) and mixed casing (`LOVE`)

This noise mirrors what you'd actually scrape from the web, and gives our cleaning step
something meaningful to do.


In [5]:
positive_templates = [
    "I absolutely LOVE this {product}!! Best purchase ever <br> Check it out: http://shop.example.com/{pid} 😍",
    "This {product} is amazing, works perfectly and arrived early. 5 stars!!! www.example.com/review{pid}",
    "Great {product}, exceeded my expectations. Highly recommend to everyone!!! <p>Buy now</p>",
    "Superb quality {product}. I am extremely satisfied with this purchase :) http://bit.ly/{pid}",
    "Fantastic {product}!! Fast shipping, great packaging, and it just WORKS. Loved it 100%.",
]

negative_templates = [
    "Terrible {product}, broke after 2 days!! Waste of money <br> Do NOT buy: http://shop.example.com/{pid}",
    "Very disappointed with this {product}. It stopped working and support was useless :( www.example.com/{pid}",
    "Awful experience, the {product} arrived damaged and customer service ignored me!!! <p>Avoid</p>",
    "Poor quality {product}. Not worth the price at all. Regret buying it http://bit.ly/{pid}",
    "Horrible {product}!! Completely malfunctioned within a week. Extremely unsatisfied 😡",
]

products = ["phone", "laptop", "headphones", "watch", "camera", "speaker", "charger", "tablet"]

rows = []
for i in range(60):
    product = random.choice(products)
    pid = random.randint(1000, 9999)
    if i % 2 == 0:
        text = random.choice(positive_templates).format(product=product, pid=pid)
        label = "positive"
    else:
        text = random.choice(negative_templates).format(product=product, pid=pid)
        label = "negative"
    rows.append({"review_id": i + 1, "raw_text": text, "sentiment": label})

df = pd.DataFrame(rows)
print(f"Generated {len(df)} synthetic reviews")
df.head(5)


Generated 60 synthetic reviews


,review_id,raw_text,sentiment
0,1,"This camera is amazing, works perfectly and ar...",positive
1,2,Poor quality phone. Not worth the price at all...,negative
2,3,I absolutely LOVE this speaker!! Best purchase...,positive
3,4,Horrible camera!! Completely malfunctioned wit...,negative
4,5,"Great charger, exceeded my expectations. Highl...",positive


In [10]:
pd.set_option('display.max_colwidth', None)
df.head(5)

,review_id,raw_text,sentiment
0,1,"This camera is amazing, works perfectly and arrived early. 5 stars!!! www.example.com/review4728",positive
1,2,Poor quality phone. Not worth the price at all. Regret buying it http://bit.ly/4164,negative
2,3,I absolutely LOVE this speaker!! Best purchase ever <br> Check it out: http://shop.example.com/5564 😍,positive
3,4,Horrible camera!! Completely malfunctioned within a week. Extremely unsatisfied 😡,negative
4,5,"Great charger, exceeded my expectations. Highly recommend to everyone!!! <p>Buy now</p>",positive


## 2. Text Cleaning

Now we clean the **`raw_text`** column. Cleaning removes structural noise that carries no
linguistic meaning:

| Noise type | Example | Regex idea |
|---|---|---|
| HTML tags | `<br>`, `<p>` | `<.*?>` |
| URLs | `http://...`, `www...` | `http\S+\|www\.\S+` |
| Special characters / punctuation | `!!!`, `:)`, emojis | `[^A-Za-z0-9\s]` |
| Extra whitespace | double spaces from removed text | `\s+` |

We keep this as a single reusable function, `clean_text()`, so it can later be applied to
any new/unseen text (e.g., at prediction time).


In [13]:
def clean_text(text):
    """Remove HTML tags, URLs, and non-alphanumeric characters; collapse whitespace."""
    text = re.sub(r"<.*?>", " ", text)                 # remove HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # remove URLs
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)         # remove punctuation/emojis/symbols
    text = re.sub(r"\s+", " ", text).strip()            # collapse multiple spaces
    return text

df["cleaned_text"] = df["raw_text"].apply(clean_text)

# Compare a couple of examples before vs. after cleaning
df[["raw_text", "cleaned_text"]].head(10)


,raw_text,cleaned_text
0,"This camera is amazing, works perfectly and arrived early. 5 stars!!! www.example.com/review4728",This camera is amazing works perfectly and arrived early 5 stars
1,Poor quality phone. Not worth the price at all. Regret buying it http://bit.ly/4164,Poor quality phone Not worth the price at all Regret buying it
2,I absolutely LOVE this speaker!! Best purchase ever <br> Check it out: http://shop.example.com/5564 😍,I absolutely LOVE this speaker Best purchase ever Check it out
3,Horrible camera!! Completely malfunctioned within a week. Extremely unsatisfied 😡,Horrible camera Completely malfunctioned within a week Extremely unsatisfied
4,"Great charger, exceeded my expectations. Highly recommend to everyone!!! <p>Buy now</p>",Great charger exceeded my expectations Highly recommend to everyone Buy now
5,"Awful experience, the phone arrived damaged and customer service ignored me!!! <p>Avoid</p>",Awful experience the phone arrived damaged and customer service ignored me Avoid
6,I absolutely LOVE this headphones!! Best purchase ever <br> Check it out: http://shop.example.com/5349 😍,I absolutely LOVE this headphones Best purchase ever Check it out
7,"Awful experience, the laptop arrived damaged and customer service ignored me!!! <p>Avoid</p>",Awful experience the laptop arrived damaged and customer service ignored me Avoid
8,"Fantastic speaker!! Fast shipping, great packaging, and it just WORKS. Loved it 100%.",Fantastic speaker Fast shipping great packaging and it just WORKS Loved it 100
9,Horrible laptop!! Completely malfunctioned within a week. Extremely unsatisfied 😡,Horrible laptop Completely malfunctioned within a week Extremely unsatisfied


In [12]:
for i, text in enumerate(df['raw_text']):
    print(f"Review {i}:")
    print(text)
    print("-" * 50)

Review 0:
This camera is amazing, works perfectly and arrived early. 5 stars!!! www.example.com/review4728
--------------------------------------------------
Review 1:
Poor quality phone. Not worth the price at all. Regret buying it http://bit.ly/4164
--------------------------------------------------
Review 2:
I absolutely LOVE this speaker!! Best purchase ever <br> Check it out: http://shop.example.com/5564 😍
--------------------------------------------------
Review 3:
Horrible camera!! Completely malfunctioned within a week. Extremely unsatisfied 😡
--------------------------------------------------
Review 4:
Great charger, exceeded my expectations. Highly recommend to everyone!!! <p>Buy now</p>
--------------------------------------------------
Review 5:
Awful experience, the phone arrived damaged and customer service ignored me!!! <p>Avoid</p>
--------------------------------------------------
Review 6:
I absolutely LOVE this headphones!! Best purchase ever <br> Check it out: http:

## 3. Preprocessing

With the noise gone, we move through the standard preprocessing chain described in the notes:

1. **Lowercasing** — normalize case so "Great" and "great" are treated identically.
2. **Tokenization** — split each sentence into individual word tokens.
3. **Stopword removal** — drop common low-information words ("the", "is", "and"...).
4. **Punctuation/non-alphabetic token removal** — drop any leftover single symbols/numbers.

Each step adds a new column so you can see exactly how the text transforms stage by stage.


In [4]:
# 3a. Lowercasing
df["lower_text"] = df["cleaned_text"].str.lower()

# 3b. Tokenization (splitting sentence into word tokens)
df["tokens"] = df["lower_text"].apply(word_tokenize)

df[["cleaned_text", "lower_text", "tokens"]].head(3)


,cleaned_text,lower_text,tokens
0,Great laptop exceeded my expectations Highly r...,great laptop exceeded my expectations highly r...,"[great, laptop, exceeded, my, expectations, hi..."
1,Very disappointed with this watch It stopped w...,very disappointed with this watch it stopped w...,"[very, disappointed, with, this, watch, it, st..."
2,I absolutely LOVE this laptop Best purchase ev...,i absolutely love this laptop best purchase ev...,"[i, absolutely, love, this, laptop, best, purc..."


In [5]:
# 3c. Stopword removal
stop_words = set(stopwords.words("english"))
print(f"Sample stopwords: {list(stop_words)[:10]}")

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)

# 3d. Remove leftover punctuation/non-alphabetic tokens (e.g., stray numbers)
def keep_alpha_tokens(tokens):
    return [t for t in tokens if t not in string.punctuation and t.isalpha()]

df["tokens_clean"] = df["tokens_no_stop"].apply(keep_alpha_tokens)

df[["tokens", "tokens_no_stop", "tokens_clean"]].head(3)


Sample stopwords: ['ma', 'some', 'further', 's', 'as', 'have', "he'd", 'you', 'those', 'we']


,tokens,tokens_no_stop,tokens_clean
0,"[great, laptop, exceeded, my, expectations, hi...","[great, laptop, exceeded, expectations, highly...","[great, laptop, exceeded, expectations, highly..."
1,"[very, disappointed, with, this, watch, it, st...","[disappointed, watch, stopped, working, suppor...","[disappointed, watch, stopped, working, suppor..."
2,"[i, absolutely, love, this, laptop, best, purc...","[absolutely, love, laptop, best, purchase, eve...","[absolutely, love, laptop, best, purchase, eve..."


## 4. Stemming vs. Lemmatization

Both reduce words to a root form, but differ in method and output quality (see the notes'
comparison table). Let's apply **both** to the same tokens and compare them side by side.

- **Stemming** (Porter Stemmer): fast, rule-based chopping — may produce non-words.
- **Lemmatization** (WordNet Lemmatizer): dictionary-based — always produces a valid word.


In [6]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

df["stemmed"] = df["tokens_clean"].apply(lambda toks: [stemmer.stem(t) for t in toks])
df["lemmatized"] = df["tokens_clean"].apply(lambda toks: [lemmatizer.lemmatize(t) for t in toks])

# Side-by-side comparison for the first 5 reviews
comparison_df = df[["tokens_clean", "stemmed", "lemmatized"]].head(5)
comparison_df


,tokens_clean,stemmed,lemmatized
0,"[great, laptop, exceeded, expectations, highly...","[great, laptop, exceed, expect, highli, recomm...","[great, laptop, exceeded, expectation, highly,..."
1,"[disappointed, watch, stopped, working, suppor...","[disappoint, watch, stop, work, support, useless]","[disappointed, watch, stopped, working, suppor..."
2,"[absolutely, love, laptop, best, purchase, eve...","[absolut, love, laptop, best, purchas, ever, c...","[absolutely, love, laptop, best, purchase, eve..."
3,"[terrible, charger, broke, days, waste, money,...","[terribl, charger, broke, day, wast, money, buy]","[terrible, charger, broke, day, waste, money, ..."
4,"[laptop, amazing, works, perfectly, arrived, e...","[laptop, amaz, work, perfectli, arriv, earli, ...","[laptop, amazing, work, perfectly, arrived, ea..."


In [7]:
# Quick word-level illustration of stemming vs lemmatization differences
sample_words = ["studies", "better", "running", "flies", "happily", "was"]

for w in sample_words:
    print(f"{w:10s} -> stem: {stemmer.stem(w):10s} | lemma: {lemmatizer.lemmatize(w)}")


studies    -> stem: studi      | lemma: study
better     -> stem: better     | lemma: better
running    -> stem: run        | lemma: running
flies      -> stem: fli        | lemma: fly
happily    -> stem: happili    | lemma: happily
was        -> stem: wa         | lemma: wa


## 5. Building the Final Preprocessed Text

For feature extraction, most vectorizers expect a **single string per document** (not a list
of tokens). We join the lemmatized tokens back into a cleaned sentence — this becomes our
final preprocessed text column, ready for numerical feature extraction.


In [8]:
df["final_text"] = df["lemmatized"].apply(lambda toks: " ".join(toks))

# Full before/after comparison: raw text vs. fully preprocessed text
df[["raw_text", "final_text", "sentiment"]].head(5)


,raw_text,final_text,sentiment
0,"Great laptop, exceeded my expectations. Highly...",great laptop exceeded expectation highly recom...,positive
1,Very disappointed with this watch. It stopped ...,disappointed watch stopped working support use...,negative
2,I absolutely LOVE this laptop!! Best purchase ...,absolutely love laptop best purchase ever check,positive
3,"Terrible charger, broke after 2 days!! Waste o...",terrible charger broke day waste money buy,negative
4,"This laptop is amazing, works perfectly and ar...",laptop amazing work perfectly arrived early star,positive


## 6. Feature Extraction

Machine learning models need **numbers**, not words. We'll build two classic representations:

### 6a. Bag of Words (BoW)
Counts how many times each word (from a fixed vocabulary) appears in each document.

### 6b. TF-IDF
Weighs each word by how important/distinctive it is to a document *relative to the whole
corpus* (see the TF-IDF formula in the notes).

We cap the vocabulary at 20 features here just so the resulting table is easy to read —
in real projects you'd typically allow a much larger vocabulary.


In [9]:
# 6a. Bag of Words
count_vec = CountVectorizer(max_features=20)
bow_matrix = count_vec.fit_transform(df["final_text"])

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vec.get_feature_names_out())
print("Bag-of-Words matrix shape:", bow_df.shape)
bow_df.head()


Bag-of-Words matrix shape: (60, 20)


,arrived,best,buy,camera,disappointed,ever,everyone,exceeded,expectation,extremely,fantastic,fast,great,headphone,highly,laptop,purchase,quality,watch,work
0,0,0,1,0,0,0,1,1,1,0,0,0,1,0,1,1,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1


In [10]:
# 6b. TF-IDF
tfidf_vec = TfidfVectorizer(max_features=20)
tfidf_matrix = tfidf_vec.fit_transform(df["final_text"])

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())
print("TF-IDF matrix shape:", tfidf_df.shape)
tfidf_df.head()


TF-IDF matrix shape: (60, 20)


,arrived,best,buy,camera,disappointed,ever,everyone,exceeded,expectation,extremely,fantastic,fast,great,headphone,highly,laptop,purchase,quality,watch,work
0,0.000000,0.000000,0.320542,0.0,0.000000,0.000000,0.4044,0.4044,0.4044,0.0,0.0,0.0,0.320542,0.0,0.4044,0.374632,0.000000,0.0,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.0,0.745163,0.000000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0000,0.000000,0.000000,0.0,0.666883,0.000000
2,0.000000,0.535559,0.000000,0.0,0.000000,0.535559,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0000,0.496136,0.424503,0.0,0.000000,0.000000
3,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0000,0.000000,0.000000,0.0,0.000000,0.000000
4,0.591522,0.000000,0.000000,0.0,0.000000,0.000000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0000,0.570132,0.000000,0.0,0.000000,0.570132


**What to notice:** in the BoW table, values are raw counts (whole numbers). In the
TF-IDF table, values are weighted floats — words that are common across *all* reviews (like
generic product words) get down-weighted, while distinctive words are weighted higher.


## 7. Bonus: Putting It All Together — A Simple Sentiment Classifier

To show *why* this whole pipeline matters, let's train a small **Multinomial Naive Bayes**
classifier on our TF-IDF features to predict `sentiment` (positive/negative). This mirrors
the **Modeling** stage of the NLP workflow from the notes.

This is only a toy example (tiny synthetic dataset), but it demonstrates the full journey:
**raw text → cleaned text → preprocessed tokens → numerical features → trained model → prediction**.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, df["sentiment"], test_size=0.25, random_state=42, stratify=df["sentiment"]
)

model = MultinomialNB()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))


Accuracy: 0.9333333333333333

              precision    recall  f1-score   support

    negative       0.89      1.00      0.94         8
    positive       1.00      0.86      0.92         7

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15



In [12]:
# Try it on a brand-new, unseen sentence — running it through the SAME pipeline steps
def preprocess_new_text(text):
    text = clean_text(text).lower()
    tokens = word_tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = keep_alpha_tokens(tokens)
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

new_review = "This laptop is absolutely amazing and works great! <br> http://shop.example.com/9999"
processed = preprocess_new_text(new_review)
vectorized = tfidf_vec.transform([processed])
prediction = model.predict(vectorized)

print("Raw input   :", new_review)
print("Preprocessed:", processed)
print("Prediction  :", prediction[0])


Raw input   : This laptop is absolutely amazing and works great! <br> http://shop.example.com/9999
Preprocessed: laptop absolutely amazing work great
Prediction  : positive


## 8. Summary

| Stage | What we did | Output column |
|---|---|---|
| Text Collection | Generated synthetic noisy reviews | `raw_text` |
| Cleaning | Removed HTML, URLs, symbols | `cleaned_text` |
| Lowercasing | Normalized case | `lower_text` |
| Tokenization | Split into word tokens | `tokens` |
| Stopword removal | Dropped low-info words | `tokens_no_stop` |
| Punctuation/token cleanup | Kept only alphabetic tokens | `tokens_clean` |
| Stemming / Lemmatization | Reduced words to root form | `stemmed`, `lemmatized` |
| Final text | Rejoined tokens into a string | `final_text` |
| Feature Extraction | Converted text to numeric vectors | BoW / TF-IDF matrices |
| Modeling | Trained + evaluated a Naive Bayes classifier | predictions |

This mirrors exactly the **Text Collection → Preprocessing → Feature Extraction → Modeling**
workflow covered in the notes. Try changing the templates in Section 1, or swapping
`MultinomialNB` for another classifier, to see how the results change!
